In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
data = pd.read_excel(
    "../datasets/basket_data.xlsx"
)

data.head()

,InvoiceID,ProductName
0,INV-2026-1001,Merino Wool Minimalist Crewneck Knitwear
1,INV-2026-1001,Bio-Peptide Restorative Night Crème
2,INV-2026-1001,Velocity Pro Carbon Running Sneakers
3,INV-2026-1001,Ergonomic Breathable Mesh Desk Chair
4,INV-2026-1001,All-Terrain 4-Season Ultralight Tent


In [3]:
print("Columns:")
print(data.columns.tolist())

print()
print("Rows:", len(data))

Columns:
['InvoiceID', 'ProductName']

Rows: 11970


In [4]:
print(data.isnull().sum())

InvoiceID      0
ProductName    0
dtype: int64


In [5]:
data = data.dropna(
    subset=["InvoiceID", "ProductName"]
).copy()

print("Rows after removing missing values:", len(data))

Rows after removing missing values: 11970


In [6]:
data["InvoiceID"] = data["InvoiceID"].astype(str)
data["ProductName"] = data["ProductName"].astype(str)

In [7]:
data = data.drop_duplicates(
    subset=["InvoiceID", "ProductName"]
)

print("Rows after removing duplicates:", len(data))

Rows after removing duplicates: 11970


In [8]:
baskets = (
    data.groupby("InvoiceID")["ProductName"]
    .apply(list)
    .tolist()
)

print("Number of baskets:", len(baskets))

print()
print("First basket:")
print(baskets[0])

Number of baskets: 2570

First basket:
['Merino Wool Minimalist Crewneck Knitwear', 'Bio-Peptide Restorative Night Crème', 'Velocity Pro Carbon Running Sneakers', 'Ergonomic Breathable Mesh Desk Chair', 'All-Terrain 4-Season Ultralight Tent']


In [9]:
encoder = TransactionEncoder()

basket_array = encoder.fit(
    baskets
).transform(baskets)

basket_df = pd.DataFrame(
    basket_array,
    columns=encoder.columns_
)

basket_df.head()

,Acacia Wood Breakfast Tray,ActiveSense GPS Smart Band,Adventure Dome Camping Shelter,AeroFlex Performance Training Top,AeroFlex Seamless High-Rise Leggings,AeroStride Performance Running Shoes,AeroTrail Waterproof Hiking Backpack,AeroWatch AMOLED Fitness Edition,AirTone Compact Wireless Earbuds,All-Terrain 4-Season Ultralight Tent,...,Urban Classic Linen Overshirt,UrbanFlex Everyday Trainers,Velocity Lightweight Trail Sneakers,Velocity Pro Carbon Running Sneakers,Velvet Rose Evening Parfum,Viper Wireless 8K Hz Esports Mouse,VitaTrack Health Smartwatch,Vitamin C Brightening Moisturizer,WarmGlow Minimalist Wall Sconce,White Tea Botanical Fragrance
0,False,False,False,False,False,False,False,False,False,True,...,False,False,False,True,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [10]:
print("Basket table shape:")
print(basket_df.shape)

print()
print(basket_df.head())

Basket table shape:
(2570, 121)

   Acacia Wood Breakfast Tray  ActiveSense GPS Smart Band  \
0                       False                       False   
1                       False                       False   
2                       False                       False   
3                       False                       False   
4                       False                       False   

   Adventure Dome Camping Shelter  AeroFlex Performance Training Top  \
0                           False                              False   
1                           False                              False   
2                           False                              False   
3                           False                              False   
4                           False                              False   

   AeroFlex Seamless High-Rise Leggings  AeroStride Performance Running Shoes  \
0                                 False                                 False   
1    

In [11]:
frequent_itemsets = apriori(
    basket_df,
    min_support=0.01,
    use_colnames=True
)

print(
    "Frequent itemsets:",
    len(frequent_itemsets)
)

Frequent itemsets: 122


In [12]:
frequent_itemsets.head(20)

,support,itemsets
0,0.041634,frozenset({Acacia Wood Breakfast Tray})
1,0.039300,frozenset({ActiveSense GPS Smart Band})
2,0.078210,frozenset({Adventure Dome Camping Shelter})
3,0.033463,frozenset({AeroFlex Performance Training Top})
4,0.032685,frozenset({AeroFlex Seamless High-Rise Leggings})
5,0.040856,frozenset({AeroStride Performance Running Shoes})
6,0.082879,frozenset({AeroTrail Waterproof Hiking Backpack})
7,0.033463,frozenset({AeroWatch AMOLED Fitness Edition})
8,0.035409,frozenset({AirTone Compact Wireless Earbuds})
9,0.091051,frozenset({All-Terrain 4-Season Ultralight Tent})


In [13]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.20
)

print(
    "Association rules:",
    len(rules)
)

Association rules: 0


In [14]:
rules_view = rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].copy()

rules_view.head(20)

,antecedents,consequents,support,confidence,lift


In [15]:
rules_view = rules_view.sort_values(
    "lift",
    ascending=False
)

rules_view.head(20)

,antecedents,consequents,support,confidence,lift


In [16]:
rules_to_save = rules.copy()

rules_to_save["antecedents"] = (
    rules_to_save["antecedents"]
    .apply(
        lambda items:
        ", ".join(sorted(map(str, items)))
    )
)

rules_to_save["consequents"] = (
    rules_to_save["consequents"]
    .apply(
        lambda items:
        ", ".join(sorted(map(str, items)))
    )
)

rules_to_save.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [17]:
rules_to_save.to_csv(
    "../backend/ml/market_basket/association_rules.csv",
    index=False
)

print("Association rules saved successfully.")

Association rules saved successfully.


In [18]:
print("=" * 60)
print("MARKET BASKET ANALYSIS COMPLETED")
print("=" * 60)

print("Transactions:", len(baskets))
print(
    "Frequent itemsets:",
    len(frequent_itemsets)
)
print(
    "Association rules:",
    len(rules)
)

print()
print("Saved file:")
print(
    "../backend/ml/market_basket/association_rules.csv"
)

MARKET BASKET ANALYSIS COMPLETED
Transactions: 2570
Frequent itemsets: 122
Association rules: 0

Saved file:
../backend/ml/market_basket/association_rules.csv
